# HDFS HAのローリングアップグレード — NameNode

StandbyからNameNodeを更新し、計画フェイルオーバー後に旧Activeを更新する。

## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*

# 操作対象クラスタの設定

D10で作成したinventoryとgroup_varsを使用する。

In [ ]:
target_group = 'hadoop_all_testcluster'
target_group

In [ ]:
!ansible {target_group} -m ping

In [ ]:
import yaml
with open('group_vars/' + target_group) as configuration:
    kerberos_enabled = 'kerberos_realm' in yaml.safe_load(configuration)
kerberos_enabled

# 管理操作の認証

Kerberos構成の場合、HDFSの管理操作を行うホストでhdfsユーザーのkeytabを使って認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

# Playbookの準備

In [ ]:
import os
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

使用するNotebookと同じ版のPlaybookを取得する。

In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1

In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

In [ ]:
active_namenode_host = 'XXX.XXX.XXX.70'
standby_namenode_host = 'XXX.XXX.XXX.71'
active_namenode_id = 'nn1'
standby_namenode_id = 'nn2'
(active_namenode_host, standby_namenode_host, active_namenode_id, standby_namenode_id)

出力とinventoryを照合し、現在のActive/Standbyを指定する。

In [ ]:
with open('group_vars/' + target_group) as configuration:
    settings = yaml.safe_load(configuration)
hadoop_dir = settings.get('hadoop_dir', '/opt/hadoop')
hadoop_package_filename = settings['hadoop_package_filename']
(hadoop_dir, hadoop_package_filename)

In [ ]:
namenode_rpc_port = 8020
namenode_rpc_port

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -rollingUpgrade prepare' -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -rollingUpgrade query' -l {target_group}

Proceed with rolling upgradeと表示されるまでqueryを再実行する。

# Standby NameNodeの更新

In [ ]:
!ansible-playbook {playbook_dir}/stop_namenode.yml -l {standby_namenode_host}

更新時は`-rollingUpgrade started`を付けて起動する。この起動引数を指定するため、更新期間中は一時サービスを使い、確定後に通常のサービスへ戻す。確定まではホストを再起動しない。

In [ ]:
!ansible {standby_namenode_host} --become -a "systemd-run --collect --unit=hadoop-hdfs-namenode-upgrade --property=User=hdfs --property=EnvironmentFile=/etc/default/hadoop-hdfs-namenode /usr/bin/env HADOOP_HOME={hadoop_dir}/{hadoop_package_filename} {hadoop_dir}/{hadoop_package_filename}/bin/hdfs --config /etc/hadoop/conf namenode -rollingUpgrade started" -l {target_group}

In [ ]:
!ansible {standby_namenode_host} --become -a "systemctl status --no-pager hadoop-hdfs-namenode-upgrade" -l {target_group}

In [ ]:
!ansible {standby_namenode_host} -m wait_for -a "host={standby_namenode_host} port={namenode_rpc_port} timeout=120" -l {target_group}

In [ ]:
!ansible {standby_namenode_host} --become -m service -a "name=hadoop-hdfs-zkfc state=started" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -getServiceState {standby_namenode_id}" -l {target_group}

standbyであり、checkHealthが成功することを確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -checkHealth {standby_namenode_id}" -l {target_group}

# 計画フェイルオーバー

更新したStandbyへ切り替える。HDFS利用者の処理が継続していることも確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -failover {active_namenode_id} {standby_namenode_id}" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

更新したNameNodeがActive、旧ActiveがStandbyであることを確認する。

# 旧Active NameNodeの更新

In [ ]:
!ansible-playbook {playbook_dir}/stop_namenode.yml -l {active_namenode_host}

In [ ]:
!ansible {active_namenode_host} --become -a "systemd-run --collect --unit=hadoop-hdfs-namenode-upgrade --property=User=hdfs --property=EnvironmentFile=/etc/default/hadoop-hdfs-namenode /usr/bin/env HADOOP_HOME={hadoop_dir}/{hadoop_package_filename} {hadoop_dir}/{hadoop_package_filename}/bin/hdfs --config /etc/hadoop/conf namenode -rollingUpgrade started" -l {target_group}

In [ ]:
!ansible {active_namenode_host} --become -a "systemctl status --no-pager hadoop-hdfs-namenode-upgrade" -l {target_group}

In [ ]:
!ansible {active_namenode_host} -m wait_for -a "host={active_namenode_host} port={namenode_rpc_port} timeout=120" -l {target_group}

In [ ]:
!ansible {active_namenode_host} --become -m service -a "name=hadoop-hdfs-zkfc state=started" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -getServiceState {active_namenode_id}" -l {target_group}

standbyであり、checkHealthが成功することを確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -checkHealth {active_namenode_id}" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

# 稼働中の版の確認

NameNodeのVersionが更新先に一致することを確認する。HTTP/HTTPSポートとCA証明書は対象環境に合わせる。

In [ ]:
namenode_http_port = 50070
namenode_http_port

In [ ]:
namenode_https_port = 50470
namenode_ca_file = '/etc/ssl/certs/ca-certificates.crt'
(namenode_https_port, namenode_ca_file)

In [ ]:
!ansible hadoop_namenode --become -m package -a "name=curl state=present" -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible hadoop_namenode --become --become-user hdfs -m shell -a 'curl --fail --no-progress-meter --negotiate -u : --cacert {namenode_ca_file} "https://$(hostname):{namenode_https_port}/jmx?qry=Hadoop:service=NameNode,name=NameNodeInfo"' -l {target_group}
else:
    !ansible hadoop_namenode --become --become-user hdfs -m shell -a 'curl --fail --no-progress-meter "http://$(hostname):{namenode_http_port}/jmx?qry=Hadoop:service=NameNode,name=NameNodeInfo"' -l {target_group}

両NameNodeの更新後、[共有Hadoop配布物の更新](O12h_Hadoop%20-%20Upgrade%20Hadoop%20node.ipynb)へ進む。